In [1]:
import torch

torch.manual_seed(42)

# Configuration
B, T, D = 2, 4, 8
BASE = 10000.0

# Sample Q and K tensors: [batch, sequence, head dimension]
q = torch.randn(B, T, D)
k = torch.randn(B, T, D)


# Build RoPE frequencies
def precompute_rope_frequencies(
    seq_len: int,
    head_dim: int,
    base: float = 10000.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Precompute cosine and sine frequencies used by RoPE."""

    theta = 1.0 / (
        base ** (
            torch.arange(0, head_dim, 2).float() / head_dim
        )
    )

    positions = torch.arange(seq_len).float()

    angles = torch.outer(positions, theta)

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# Apply the rotary transformation to Q or K
def apply_rope(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    """Apply rotary positional embeddings to the last dimension."""

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_even * sin + x_odd * cos

    x_rotated = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    )

    return x_rotated.flatten(-2)


# Precompute positional frequencies
cos, sin = precompute_rope_frequencies(
    seq_len=T,
    head_dim=D,
    base=BASE,
)

# Reshape frequencies for broadcasting over the batch dimension
cos = cos.unsqueeze(0)
sin = sin.unsqueeze(0)

# Apply RoPE to Q and K
q_rope = apply_rope(q, cos, sin)
k_rope = apply_rope(k, cos, sin)

# Verify shapes
print("Q shape:      ", q.shape)
print("K shape:      ", k.shape)
print("Cos shape:    ", cos.shape)
print("Sin shape:    ", sin.shape)
print("Q RoPE shape: ", q_rope.shape)
print("K RoPE shape: ", k_rope.shape)

# Verify that RoPE changes the representations
print("\nQ changed:", not torch.allclose(q, q_rope))
print("K changed:", not torch.allclose(k, k_rope))

Q shape:       torch.Size([2, 4, 8])
K shape:       torch.Size([2, 4, 8])
Cos shape:     torch.Size([1, 4, 4])
Sin shape:     torch.Size([1, 4, 4])
Q RoPE shape:  torch.Size([2, 4, 8])
K RoPE shape:  torch.Size([2, 4, 8])

Q changed: True
K changed: True


In [2]:
# Position 0 has zero rotation because its angle is zero
print("Q position 0 before:")
print(q[0, 0])

print("\nQ position 0 after:")
print(q_rope[0, 0])

print("\nPosition 0 unchanged:", torch.allclose(q[0, 0], q_rope[0, 0]))

Q position 0 before:
tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047])

Q position 0 after:
tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047])

Position 0 unchanged: True
